In [2]:
"""
Script COMBINADO para construir la tabla GHZ completa de un backend,
promediando dos tandas (v1 y v2), CUANDO TODO (results_*.json Y los
qubit_properties_*.pkl de calibracion de TODOS los backends) esta
mezclado dentro de las mismas dos carpetas v1/ y v2/.

Que hace:
    - Busca dentro de V1_DIR / V2_DIR los results_*.json del backend
      indicado en BACKEND_FILTER -> saca "fidelity"
    - Busca dentro de las MISMAS carpetas los qubit_properties_*.pkl
      del MISMO backend -> si hay varios (distintas fechas), elige
      automaticamente el que tenga el sello de fecha mas cercano a
      los resultados de esa carpeta
    - Con ese .pkl, RECONSTRUYE el circuito GHZ transpilado para cada
      tamaño de qubit y mide circuit_depth y two_qubit_gates
    - Promedia v1 y v2 para cada tamaño de qubit
    - Calcula threshold (0.5 - epsilon) y Pass/Fail
    - Imprime la tabla final y la guarda en un CSV

IMPORTANTE:
    El circuit_depth / two_qubit_gates son una RECONSTRUCCION (se vuelve a
    transpilar el circuito GHZ contra la calibracion guardada en el .pkl
    mas cercano en fecha), no el dato exacto registrado durante la
    ejecucion original.

USO:
    1. Rellena V1_DIR, V2_DIR y BACKEND_FILTER mas abajo.
    2. Ejecuta:  python build_full_ghz_table.py
    3. Te imprime la tabla final y guarda 'full_ghz_table.csv'
"""

import json
import os
import re
import glob
import csv
import pickle
from datetime import datetime

from qiskit import QuantumCircuit
from qiskit.transpiler import generate_preset_pass_manager


# -------------------------------------------------------------------
# 1) CONFIGURA AQUI TUS RUTAS
# -------------------------------------------------------------------
V1_DIR = "./v1"   # carpeta con TODOS los results_*.json Y *.pkl (tanda 1)
V2_DIR = "./v2"   # carpeta con TODOS los results_*.json Y *.pkl (tanda 2)

BACKEND_FILTER = "ibm_torino"   # nombre del backend a procesar

GHZ_MODE = "lineal"          # ver utils/circuits.py si usasteis otro modo
OPTIMIZATION_LEVEL = 1       # nivel de optimizacion al transpilar

OUTPUT_CSV = "full_ghz_table.csv"


# -------------------------------------------------------------------
# 2) LECTURA LIGERA DE LOS results_*.json (sin cargar arrays pesados)
# -------------------------------------------------------------------
FILENAME_PATTERN = re.compile(r"results_(.+?)_(\d+)q_(\d{8}_\d{6})\.json")


def read_light_fields(filepath):
    fields = {"backend": None, "numero_qubits_inicial": None,
              "epsilon": None, "delta": None, "fidelity": None}

    with open(filepath, "r") as f:
        head = f.read(4000)

    def grab_scalar(key, cast=str):
        m = re.search(rf'"{key}":\s*([^,\n\}}]+)', head)
        if not m:
            return None
        val = m.group(1).strip().strip('"')
        try:
            return cast(val)
        except Exception:
            return val

    fields["backend"] = grab_scalar("backend")
    fields["numero_qubits_inicial"] = grab_scalar("numero_qubits_inicial", int)
    fields["epsilon"] = grab_scalar("epsilon", float)
    fields["delta"] = grab_scalar("delta", float)

    filesize = os.path.getsize(filepath)
    with open(filepath, "rb") as f:
        f.seek(max(0, filesize - 300))
        tail = f.read().decode(errors="ignore")
    m = re.search(r'"fidelity":\s*([^\s}]+)', tail)
    if m:
        fields["fidelity"] = float(m.group(1))
    else:
        with open(filepath, "r") as f:
            content = f.read()
        m = re.search(r'"fidelity":\s*([^\s,}]+)', content)
        if m:
            fields["fidelity"] = float(m.group(1))

    return fields


def scan_folder(folder):
    records = []
    for filepath in sorted(glob.glob(os.path.join(folder, "results_*.json"))):
        fname = os.path.basename(filepath)
        m = FILENAME_PATTERN.match(fname)
        if not m:
            continue
        backend_in_name, qubits_in_name, timestamp = m.groups()
        if BACKEND_FILTER and backend_in_name != BACKEND_FILTER:
            continue
        fields = read_light_fields(filepath)
        fields["file"] = fname
        fields["qubits_in_name"] = int(qubits_in_name)
        fields["timestamp"] = timestamp
        records.append(fields)
    return records


def fidelity_by_qubits(records):
    """{n_qubits: {"fidelity":..., "epsilon":..., "timestamps":[...]}} promediando duplicados internos."""
    by_n = {}
    for r in records:
        by_n.setdefault(r["qubits_in_name"], []).append(r)
    out = {}
    for n, recs in by_n.items():
        fids = [r["fidelity"] for r in recs if r["fidelity"] is not None]
        epss = [r["epsilon"] for r in recs if r["epsilon"] is not None]
        out[n] = {
            "fidelity": sum(fids) / len(fids) if fids else None,
            "epsilon": sum(epss) / len(epss) if epss else None,
            "timestamps": [r["timestamp"] for r in recs],
        }
    return out


# -------------------------------------------------------------------
# 2b) LOCALIZAR EL .pkl DE CALIBRACION CORRECTO DENTRO DE v1 / v2
# -------------------------------------------------------------------
PKL_PATTERN = re.compile(r"qubit_properties_(.+?)_(\d+)q_(\d{8}_\d{6})\.pkl")


def find_calibration_pkls(folder, backend_filter):
    """
    Busca todos los qubit_properties_{backend}_{qubits}q_{timestamp}.pkl
    de un backend concreto dentro de una carpeta. Devuelve una lista de
    dicts {file, filepath, timestamp} ordenada por fecha.
    """
    found = []
    for filepath in sorted(glob.glob(os.path.join(folder, "qubit_properties_*.pkl"))):
        fname = os.path.basename(filepath)
        m = PKL_PATTERN.match(fname)
        if not m:
            continue
        backend_in_name, _qubits, timestamp = m.groups()
        if backend_in_name != backend_filter:
            continue
        found.append({"file": fname, "filepath": filepath, "timestamp": timestamp})
    found.sort(key=lambda d: d["timestamp"])
    return found


def closest_pkl(pkl_list, target_timestamp):
    """
    De una lista de calibraciones disponibles, elige la que tenga el
    timestamp mas cercano a target_timestamp (formato YYYYMMDD_HHMMSS).
    """
    if not pkl_list:
        return None
    if len(pkl_list) == 1 or target_timestamp is None:
        return pkl_list[0]

    fmt = "%Y%m%d_%H%M%S"
    target_dt = datetime.strptime(target_timestamp, fmt)

    best = min(
        pkl_list,
        key=lambda d: abs((datetime.strptime(d["timestamp"], fmt) - target_dt).total_seconds())
    )
    return best


# -------------------------------------------------------------------
# 3) RECONSTRUCCION DEL CIRCUITO GHZ (igual que utils/circuits.py)
# -------------------------------------------------------------------
def create_lineal_ghz_circuit(num_qubits):
    qc = QuantumCircuit(num_qubits)
    qc.h(0)
    for i in range(num_qubits - 1):
        qc.cx(i, i + 1)
    return qc


def create_ghz_circuit(num_qubits, mode="lineal"):
    if mode == "lineal":
        return create_lineal_ghz_circuit(num_qubits)
    raise NotImplementedError(
        f"Modo '{mode}' no incluido en esta version reducida del script; "
        "copia las demas funciones de reconstruct_depth_2qgates.py si lo necesitas."
    )


def depth_and_gates(num_qubits, target):
    qc = create_ghz_circuit(num_qubits, mode=GHZ_MODE)
    pm = generate_preset_pass_manager(optimization_level=OPTIMIZATION_LEVEL, target=target)
    transpiled = pm.run(qc)
    depth = transpiled.depth()
    two_q = sum(1 for instr in transpiled.data if instr.operation.num_qubits == 2)
    return depth, two_q


# -------------------------------------------------------------------
# 4) PROGRAMA PRINCIPAL
# -------------------------------------------------------------------
def _avg(values):
    clean = [v for v in values if v is not None]
    return sum(clean) / len(clean) if clean else None


def main():
    # --- comprobaciones basicas ---
    for path, label in [(V1_DIR, "V1_DIR"), (V2_DIR, "V2_DIR")]:
        if not os.path.isdir(path):
            print(f"ERROR: {label} no existe: {path}")
            return

    # --- fidelidad desde los JSON ---
    print(f"Leyendo fidelidad de v1: {V1_DIR}  (backend={BACKEND_FILTER})")
    v1_records = scan_folder(V1_DIR)
    v1_fid = fidelity_by_qubits(v1_records)
    print(f"  -> {len(v1_records)} archivos results_*.json, tamaños: {sorted(v1_fid.keys())}")

    print(f"Leyendo fidelidad de v2: {V2_DIR}  (backend={BACKEND_FILTER})")
    v2_records = scan_folder(V2_DIR)
    v2_fid = fidelity_by_qubits(v2_records)
    print(f"  -> {len(v2_records)} archivos results_*.json, tamaños: {sorted(v2_fid.keys())}")

    all_qubit_sizes = sorted(set(v1_fid.keys()) | set(v2_fid.keys()))
    if not all_qubit_sizes:
        print(f"ERROR: no se encontro ningun results_*.json de backend "
              f"'{BACKEND_FILTER}' en v1 ni v2. Revisa BACKEND_FILTER.")
        return

    # --- localizar los .pkl de calibracion DENTRO de v1 y v2 ---
    v1_pkls = find_calibration_pkls(V1_DIR, BACKEND_FILTER)
    v2_pkls = find_calibration_pkls(V2_DIR, BACKEND_FILTER)

    print(f"\nCalibraciones (.pkl) encontradas para '{BACKEND_FILTER}':")
    print(f"  en v1: {[p['file'] for p in v1_pkls] if v1_pkls else 'NINGUNA'}")
    print(f"  en v2: {[p['file'] for p in v2_pkls] if v2_pkls else 'NINGUNA'}")

    if not v1_pkls and not v2_pkls:
        print(f"\nERROR: no se encontro ningun qubit_properties_{BACKEND_FILTER}_*.pkl "
              f"ni en v1 ni en v2. No se puede reconstruir Depth / 2Q gates.")
        return

    # cache de targets ya cargados, para no recargar el mismo .pkl varias veces
    _target_cache = {}

    def load_target(pkl_info):
        if pkl_info is None:
            return None
        path = pkl_info["filepath"]
        if path not in _target_cache:
            with open(path, "rb") as f:
                _target_cache[path] = pickle.load(f)
        return _target_cache[path]

    # --- construir filas ---
    rows = []
    print(f"\n{'Qubits':>7} | {'Fid v1':>8} | {'Fid v2':>8} | {'MeanFid':>8} | "
          f"{'Depth v1':>9} | {'Depth v2':>9} | {'MeanDep':>8} | "
          f"{'2Qg v1':>7} | {'2Qg v2':>7} | {'Mean2Qg':>8} | {'Thresh':>7} | Pass/Fail")
    print("-" * 130)

    for n in all_qubit_sizes:
        fid_v1 = v1_fid.get(n, {}).get("fidelity")
        fid_v2 = v2_fid.get(n, {}).get("fidelity")
        mean_fid = _avg([fid_v1, fid_v2])

        eps_v1 = v1_fid.get(n, {}).get("epsilon")
        eps_v2 = v2_fid.get(n, {}).get("epsilon")
        epsilon = _avg([eps_v1, eps_v2])
        threshold = (0.5 - epsilon) if epsilon is not None else None

        # Para reconstruir depth/2Q gates en v1: solo si ese n existe en v1
        # Y hay al menos un .pkl de ese backend en v1. Elegimos el .pkl con
        # fecha mas cercana al timestamp de ESE resultado concreto.
        depth_v1 = gates_v1 = None
        if n in v1_fid and v1_pkls:
            result_ts = v1_fid[n]["timestamps"][0] if v1_fid[n]["timestamps"] else None
            chosen_pkl = closest_pkl(v1_pkls, result_ts)
            target_v1 = load_target(chosen_pkl)
            depth_v1, gates_v1 = depth_and_gates(n, target_v1)

        depth_v2 = gates_v2 = None
        if n in v2_fid and v2_pkls:
            result_ts = v2_fid[n]["timestamps"][0] if v2_fid[n]["timestamps"] else None
            chosen_pkl = closest_pkl(v2_pkls, result_ts)
            target_v2 = load_target(chosen_pkl)
            depth_v2, gates_v2 = depth_and_gates(n, target_v2)

        mean_depth = _avg([depth_v1, depth_v2])
        mean_gates = _avg([gates_v1, gates_v2])

        passed = (mean_fid is not None and threshold is not None and mean_fid > threshold)

        def fmt(x, nd=4):
            return f"{x:.{nd}f}" if isinstance(x, float) else ("--" if x is None else str(x))

        print(f"{n:>7} | {fmt(fid_v1):>8} | {fmt(fid_v2):>8} | {fmt(mean_fid):>8} | "
              f"{fmt(depth_v1,0):>9} | {fmt(depth_v2,0):>9} | {fmt(mean_depth,1):>8} | "
              f"{fmt(gates_v1,0):>7} | {fmt(gates_v2,0):>7} | {fmt(mean_gates,1):>8} | "
              f"{fmt(threshold,2):>7} | {'Pass' if passed else 'Fail'}")

        rows.append({
            "qubits": n,
            "fidelity_v1": fid_v1, "fidelity_v2": fid_v2, "mean_fidelity": mean_fid,
            "depth_v1": depth_v1, "depth_v2": depth_v2, "mean_depth": mean_depth,
            "two_qubit_gates_v1": gates_v1, "two_qubit_gates_v2": gates_v2,
            "mean_two_qubit_gates": mean_gates,
            "epsilon": epsilon, "threshold": threshold,
            "pass_fail": "Pass" if passed else "Fail",
        })

    if not rows:
        print("\nERROR: no se genero ninguna fila.")
        return

    with open(OUTPUT_CSV, "w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=list(rows[0].keys()))
        writer.writeheader()
        writer.writerows(rows)

    print(f"\nTabla guardada en: {OUTPUT_CSV}")
    print("\nRECUERDA: Depth y 2Q gates son una RECONSTRUCCION del circuito")
    print("transpilado contra la calibracion (.pkl) mas cercana en fecha a")
    print("cada resultado, no el dato exacto registrado durante la ejecucion original.")


if __name__ == "__main__":
    main()

Leyendo fidelidad de v1: ./v1  (backend=ibm_torino)
  -> 5 archivos results_*.json, tamaños: [16, 19, 20, 21, 23]
Leyendo fidelidad de v2: ./v2  (backend=ibm_torino)
  -> 5 archivos results_*.json, tamaños: [16, 19, 20, 21, 23]

Calibraciones (.pkl) encontradas para 'ibm_torino':
  en v1: ['qubit_properties_ibm_torino_133q_20260329_234422.pkl']
  en v2: ['qubit_properties_ibm_torino_133q_20260331_112619.pkl']

 Qubits |   Fid v1 |   Fid v2 |  MeanFid |  Depth v1 |  Depth v2 |  MeanDep |  2Qg v1 |  2Qg v2 |  Mean2Qg |  Thresh | Pass/Fail
----------------------------------------------------------------------------------------------------------------------------------
     16 |   0.5010 |   0.4701 |   0.4856 |        63 |        63 |     63.0 |      15 |      15 |     15.0 |    0.40 | Pass
     19 |   0.4003 |   0.4142 |   0.4073 |        75 |        75 |     75.0 |      18 |      18 |     18.0 |    0.40 | Pass
     20 |   0.3945 |   0.3887 |   0.3916 |        79 |        79 |     79.0 | 

In [4]:
"""
Script para extraer TODOS los datos de la parte superior del informe SPEQC
(bloques "Hardware" e "Identity") a partir de los .pkl de calibracion
(qubit_properties_*.pkl), buscandolos dentro de v1/ y v2/ y promediando
entre ambas tandas.

Que saca:
    - Total qubits (del backend)
    - Native gates (lista de operaciones soportadas)
    - Topology (heuristica simple: heavy-hex / square / desconocida,
      basada en el grado medio de conectividad)
    - Median T1, Median T2 (de cada tanda y la media entre ambas)
    - Error mediano y duracion mediana de la gate nativa de 2 qubits
    - Error mediano de la gate nativa de 1 qubit (si existe)
    - Fecha de la calibracion (extraida del nombre del archivo)

Lo que NO puede sacar (no existe en el objeto Target, hay que poner a mano):
    - "Calibration time required" (no se guarda en el Target)
    - "Shots per circuit" (es un parametro de ejecucion en main.py, no de
      la calibracion del backend)

USO:
    1. Rellena V1_DIR, V2_DIR, BACKEND_FILTER mas abajo.
    2. Ejecuta:  python extract_hardware_metadata.py
    3. Te imprime un resumen listo para copiar a la tabla SPEQC.
"""

import os
import re
import glob
import pickle
import statistics
from datetime import datetime


# -------------------------------------------------------------------
# 1) CONFIGURA AQUI
# -------------------------------------------------------------------
V1_DIR = "./v1"
V2_DIR = "./v2"
BACKEND_FILTER = "ibm_torino"

# Si solo tienes un .pkl (no dos, uno por tanda), deja V2 igual a V1 o
# vacio; el script se adapta.

PKL_PATTERN = re.compile(r"qubit_properties_(.+?)_(\d+)q_(\d{8}_\d{6})\.pkl")


# -------------------------------------------------------------------
# 2) LOCALIZAR LOS .pkl DEL BACKEND DENTRO DE UNA CARPETA
# -------------------------------------------------------------------
def find_calibration_pkls(folder, backend_filter):
    found = []
    if not os.path.isdir(folder):
        return found
    for filepath in sorted(glob.glob(os.path.join(folder, "qubit_properties_*.pkl"))):
        fname = os.path.basename(filepath)
        m = PKL_PATTERN.match(fname)
        if not m:
            continue
        backend_in_name, qubits, timestamp = m.groups()
        if backend_in_name != backend_filter:
            continue
        found.append({
            "file": fname, "filepath": filepath,
            "qubits": int(qubits), "timestamp": timestamp,
        })
    found.sort(key=lambda d: d["timestamp"])
    return found


def format_timestamp(ts):
    """20260330_123519 -> 2026-03-30 12:35:19"""
    try:
        dt = datetime.strptime(ts, "%Y%m%d_%H%M%S")
        return dt.strftime("%Y-%m-%d %H:%M:%S")
    except Exception:
        return ts


# -------------------------------------------------------------------
# 3) EXTRAER METADATOS DE UN .pkl (objeto Target de Qiskit)
# -------------------------------------------------------------------
def extract_metadata(pkl_path):
    with open(pkl_path, "rb") as f:
        target = pickle.load(f)

    meta = {}
    meta["num_qubits"] = target.num_qubits
    meta["operation_names"] = sorted(target.operation_names)

    # --- T1 / T2 medianos ---
    qp = target.qubit_properties
    t1_values = [q.t1 for q in qp if q.t1 is not None] if qp else []
    t2_values = [q.t2 for q in qp if q.t2 is not None] if qp else []
    meta["median_t1_us"] = (statistics.median(t1_values) * 1e6) if t1_values else None
    meta["median_t2_us"] = (statistics.median(t2_values) * 1e6) if t2_values else None

    # --- topologia: grado medio de conectividad ---
    cmap = target.build_coupling_map()
    if cmap is not None:
        n_edges = cmap.size()
        n_qubits = target.num_qubits
        avg_degree = (2 * n_edges) / n_qubits if n_qubits else None
        meta["avg_connectivity_degree"] = avg_degree
        if avg_degree is not None:
            if avg_degree < 2.3:
                meta["topology_guess"] = "Heavy-hex (low connectivity, ~2 per qubit)"
            elif avg_degree < 3.3:
                meta["topology_guess"] = "Square / heavy-square (~3 per qubit)"
            elif avg_degree < 4.5:
                meta["topology_guess"] = "Square lattice / grid (~4 per qubit)"
            else:
                meta["topology_guess"] = "High connectivity / all-to-all-like"
    else:
        meta["avg_connectivity_degree"] = None
        meta["topology_guess"] = "Unknown (no coupling map found)"

    # --- gate nativa de 2 qubits: buscamos la primera de 2 qargs que conozcamos ---
    two_q_gate_name = None
    for name in target.operation_names:
        try:
            qargs_list = [q for q in target[name].keys() if q is not None]
        except Exception:
            continue
        if qargs_list and len(qargs_list[0]) == 2:
            two_q_gate_name = name
            break

    if two_q_gate_name:
        props = target[two_q_gate_name]
        errors = [p.error for q, p in props.items() if q is not None and p is not None and p.error is not None]
        durations = [p.duration for q, p in props.items() if q is not None and p is not None and p.duration is not None]
        meta["two_qubit_gate_name"] = two_q_gate_name
        meta["two_qubit_gate_median_error"] = statistics.median(errors) if errors else None
        meta["two_qubit_gate_median_duration_ns"] = (
            statistics.median(durations) * 1e9 if durations else None
        )
    else:
        meta["two_qubit_gate_name"] = None
        meta["two_qubit_gate_median_error"] = None
        meta["two_qubit_gate_median_duration_ns"] = None

    # --- gate nativa de 1 qubit mas comun (ej. sx, x, rz) ---
    one_q_gate_name = None
    for candidate in ["sx", "x", "rz", "id"]:
        if candidate in target.operation_names:
            try:
                qargs_list = [q for q in target[candidate].keys() if q is not None]
            except Exception:
                continue
            if qargs_list and len(qargs_list[0]) == 1:
                one_q_gate_name = candidate
                break

    if one_q_gate_name:
        props = target[one_q_gate_name]
        errors = [p.error for q, p in props.items() if q is not None and p is not None and p.error is not None]
        meta["one_qubit_gate_name"] = one_q_gate_name
        meta["one_qubit_gate_median_error"] = statistics.median(errors) if errors else None
    else:
        meta["one_qubit_gate_name"] = None
        meta["one_qubit_gate_median_error"] = None

    return meta


# -------------------------------------------------------------------
# 4) PROGRAMA PRINCIPAL
# -------------------------------------------------------------------
def _avg(values):
    clean = [v for v in values if v is not None]
    return sum(clean) / len(clean) if clean else None


def report_for_folder(label, folder):
    print(f"\n--- {label}: {folder} ---")
    pkls = find_calibration_pkls(folder, BACKEND_FILTER)
    if not pkls:
        print(f"  No se encontro ningun qubit_properties_{BACKEND_FILTER}_*.pkl aqui.")
        return None

    # Si hay varios, usamos el mas reciente de esa carpeta
    chosen = pkls[-1]
    print(f"  Usando: {chosen['file']}  (fecha: {format_timestamp(chosen['timestamp'])})")
    if len(pkls) > 1:
        print(f"  (Aviso: habia {len(pkls)} archivos de este backend en esta carpeta; "
              f"se usa el mas reciente. Otros encontrados: "
              f"{[p['file'] for p in pkls if p != chosen]})")

    meta = extract_metadata(chosen["filepath"])
    meta["timestamp"] = chosen["timestamp"]
    return meta


def main():
    meta_v1 = report_for_folder("v1", V1_DIR)
    meta_v2 = report_for_folder("v2", V2_DIR)

    if meta_v1 is None and meta_v2 is None:
        print(f"\nERROR: no se encontro ningun .pkl de '{BACKEND_FILTER}' en v1 ni v2.")
        return

    print("\n" + "=" * 70)
    print("RESUMEN PARA LA TABLA SPEQC (bloque Hardware)")
    print("=" * 70)

    # --- campos que no cambian entre v1/v2 (cogemos el que exista) ---
    ref = meta_v1 or meta_v2
    print(f"Total qubits:            {ref['num_qubits']}")
    print(f"Native gates:            {', '.join(ref['operation_names'])}")
    print(f"Topology (heuristic):    {ref['topology_guess']}  "
          f"(avg degree = {ref['avg_connectivity_degree']:.2f})")

    # --- gate de 2 qubits ---
    g2 = ref["two_qubit_gate_name"]
    if g2:
        err = ref["two_qubit_gate_median_error"]
        dur = ref["two_qubit_gate_median_duration_ns"]
        print(f"2Q native gate:          {g2}  "
              f"(median error = {err*100:.3f}% , median duration = {dur:.1f} ns)")

    g1 = ref["one_qubit_gate_name"]
    if g1:
        err1 = ref["one_qubit_gate_median_error"]
        print(f"1Q native gate:          {g1}  (median error = {err1*100:.4f}%)")

    # --- T1 / T2, promediando v1 y v2 si ambos existen ---
    t1_v1 = meta_v1["median_t1_us"] if meta_v1 else None
    t1_v2 = meta_v2["median_t1_us"] if meta_v2 else None
    t2_v1 = meta_v1["median_t2_us"] if meta_v1 else None
    t2_v2 = meta_v2["median_t2_us"] if meta_v2 else None

    mean_t1 = _avg([t1_v1, t1_v2])
    mean_t2 = _avg([t2_v1, t2_v2])

    print()
    def fmt_us(x):
        return f"{x:.1f} us" if x is not None else "--"

    print(f"Median T1:               v1={fmt_us(t1_v1)}  v2={fmt_us(t1_v2)}  "
          f"-> MEAN = {fmt_us(mean_t1)}")
    print(f"Median T2:               v1={fmt_us(t2_v1)}  v2={fmt_us(t2_v2)}  "
          f"-> MEAN = {fmt_us(mean_t2)}")

    if meta_v1:
        print(f"\nFecha calibracion v1:    {format_timestamp(meta_v1['timestamp'])}")
    if meta_v2:
        print(f"Fecha calibracion v2:    {format_timestamp(meta_v2['timestamp'])}")

    print("\n" + "-" * 70)
    print("CAMPOS QUE NO SE PUEDEN SACAR DEL .pkl (rellenar a mano si los teneis):")
    print("  - Calibration time required  (no existe en el objeto Target)")
    print("  - Shots per circuit          (parametro de ejecucion en main.py)")
    print("-" * 70)


if __name__ == "__main__":
    main()


--- v1: ./v1 ---
  Usando: qubit_properties_ibm_torino_133q_20260329_234422.pkl  (fecha: 2026-03-29 23:44:22)

--- v2: ./v2 ---
  Usando: qubit_properties_ibm_torino_133q_20260331_112619.pkl  (fecha: 2026-03-31 11:26:19)

RESUMEN PARA LA TABLA SPEQC (bloque Hardware)
Total qubits:            133
Native gates:            cz, delay, id, if_else, measure, reset, rz, sx, x
Topology (heuristic):    Heavy-hex (low connectivity, ~2 per qubit)  (avg degree = 2.00)
2Q native gate:          cz  (median error = 0.252% , median duration = 68.0 ns)
1Q native gate:          sx  (median error = 0.0309%)

Median T1:               v1=175.5 us  v2=185.6 us  -> MEAN = 180.5 us
Median T2:               v1=132.3 us  v2=135.3 us  -> MEAN = 133.8 us

Fecha calibracion v1:    2026-03-29 23:44:22
Fecha calibracion v2:    2026-03-31 11:26:19

----------------------------------------------------------------------
CAMPOS QUE NO SE PUEDEN SACAR DEL .pkl (rellenar a mano si los teneis):
  - Calibration time requir